In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K

In [ ]:
train_dir = "/kaggle/input/bangladeshi-weedy-area-medicinal-plant/medicinal_weedy_area_image_dataset_version2/medicinal_weedy_area_image_dataset_version2/train"
test_dir = "/kaggle/input/bangladeshi-weedy-area-medicinal-plant/medicinal_weedy_area_image_dataset_version2/medicinal_weedy_area_image_dataset_version2/test"
val_dir = "/kaggle/input/bangladeshi-weedy-area-medicinal-plant/medicinal_weedy_area_image_dataset_version2/medicinal_weedy_area_image_dataset_version2/val"

In [ ]:
img_size = 224
batch_size = 32

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=True,
    color_mode="rgb",
    seed=42
)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=False,
    color_mode="rgb",
    seed=42
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=False,
    color_mode="rgb",
    seed=42
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")


rescale = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255)
])

def apply_rescaling(images, labels):
    return rescale(images), labels

train_dataset = train_dataset.map(apply_rescaling, num_parallel_calls=AUTOTUNE)
val_dataset = val_dataset.map(apply_rescaling, num_parallel_calls=AUTOTUNE)
test_dataset = test_dataset.map(apply_rescaling, num_parallel_calls=AUTOTUNE)


train_dataset = train_dataset.cache().prefetch(AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(AUTOTUNE)

In [ ]:
def inverted_residual_block(x_in, filters_out, strides=1, expansion_factor=4, use_attention=True, dropout_rate=0.0):
    shortcut = x_in
    filters_in = K.int_shape(x_in)[-1]
    
    if expansion_factor > 1:
        expanded_channels = filters_in * expansion_factor
        x = Conv2D(expanded_channels, 1, padding='same', use_bias=False, 
                  kernel_initializer='he_normal')(x_in)
        x = BatchNormalization()(x)
        x = Activation('relu6')(x)
    else:
        x = x_in
    
    x = DepthwiseConv2D(3, strides=strides, padding='same', use_bias=False, depthwise_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu6')(x)
    
    if use_attention and filters_out >= 64:
        x = efficient_channel_attention(x, ratio=8)
    
    x = Conv2D(filters_out, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    
    if strides == 1 and filters_in == filters_out:
        x = Add()([shortcut, x])
    
    return x

def create_flora_net_lite(input_shape, num_classes, width_multiplier=0.75):
    inputs = Input(shape=input_shape)
    x = inputs
    
    def make_divisible(v, divisor=8):
        new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
        if new_v < 0.9 * v:
            new_v += divisor
        return new_v
    
    filters = make_divisible(16 * width_multiplier)
    x = Conv2D(filters, 3, strides=2, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu6')(x)
    
    stage_configs = [
        (24, 2, 2, False),
        (32, 3, 2, False),
        (64, 4, 2, True),
        (96, 6, 1, True),
    ]
    
    for filters, expansion, stride, use_attention in stage_configs:
        filters = make_divisible(filters * width_multiplier)
        x = inverted_residual_block(x, filters, strides=stride, expansion_factor=expansion, use_attention=use_attention, dropout_rate=0.1)
        x = inverted_residual_block(x, filters, strides=1, expansion_factor=expansion, use_attention=use_attention)
    
    final_filters = make_divisible(320 * width_multiplier)
    x = Conv2D(final_filters, 1, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu6')(x)
    
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='FloraNetLite')
    return model

In [ ]:
input_shape = (img_size, img_size, 3)
model = create_flora_net_lite(input_shape, num_classes, width_multiplier=0.5)

In [ ]:
def get_callbacks(model_name):
    return [
        ModelCheckpoint(
            f'/kaggle/working/best_loss_{model_name}.keras', 
            monitor='val_loss', 
            save_best_only=True, 
            mode='min', 
            verbose=1
        ),
        ModelCheckpoint(
            f'/kaggle/working/best_accuracy_{model_name}.keras', 
            monitor='val_accuracy', 
            save_best_only=True, 
            mode='max',
            verbose=1
        ),
        ModelCheckpoint(
            f'/kaggle/working/best_model_loss_{model_name}.keras', 
            monitor='loss', 
            save_best_only=True, 
            mode='min',
            verbose=1
        ),
        EarlyStopping(
            monitor='loss',
            patience=25,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='loss',
            factor=0.5,
            patience=10,
            min_lr=1e-8,
            verbose=1
        )
    ]

In [ ]:
print("\nFloraNet Summary:")
callbacks = get_callbacks("FloraNet")
model.compile(optimizer=SGD(learning_rate=0.01, momentum=0.8, nesterov=True), loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(train_dataset, epochs=300, validation_data=val_dataset, callbacks=callbacks)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

def evaluate_model(model, history, name, test_dataset):
    print(f"\nEvaluating {name}...")
    test_loss, test_acc = model.evaluate(test_dataset)
    print(f"Test accuracy: {test_acc:.4f}")
    
    y_pred_prob = model.predict(test_dataset)
    y_pred = np.argmax(y_pred_prob, axis=1)
    

    y_true = []
    for images, labels in test_dataset.unbatch():
        if len(labels.shape) > 0 and labels.shape[0] > 1:
            y_true.append(tf.argmax(labels).numpy())
        else:
            y_true.append(int(labels.numpy()))
    
    y_true = np.array(y_true[:len(y_pred)])
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, cmap='Blues', cbar=True)
    plt.title(f"Confusion Matrix - {name}", fontsize=16)
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/confusion_matrix_{name.replace(" ", "_")}.png')
    plt.close()
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], color='darkorange', label='Training')
    plt.plot(history.history['val_loss'], color='blue', label='Validation')
    plt.title(f'Loss vs Epochs - {name}', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], color='darkorange', label='Training')
    plt.plot(history.history['val_accuracy'], color='blue', label='Validation')
    plt.title(f'Accuracy vs Epochs - {name}', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/training_history_{name.replace(" ", "_")}.png')
    plt.close()
    
    return accuracy, y_true, y_pred, y_pred_prob

In [ ]:
acc1, y_true1, y_pred1, y_pred_prob1 = evaluate_model(model, history, "FloraNet", test_dataset)

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)

In [ ]:
cm = confusion_matrix(y_true1, y_pred1, normalize='true')
sns.heatmap(cm, annot=True, cmap='Blues', fmt=".2f")


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle

# Binarize the true labels
y_true_bin = label_binarize(y_true1, classes=list(range(num_classes)))

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob1[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot all ROC curves
plt.figure(figsize=(12, 10))
colors = cycle(plt.cm.tab20.colors)

for i, color in zip(range(num_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"Class {class_names[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve')
plt.legend(loc="lower right", fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curve_multiclass.png')
plt.close()

In [ ]:
pip install lime --quiet


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import random

explainer = lime_image.LimeImageExplainer()

# Pick a sample image from the test dataset
for image_batch, label_batch in test_dataset.take(1):
    idx = random.randint(0, batch_size - 1)
    image = image_batch[idx].numpy()
    label = label_batch[idx].numpy()
    break

# Explaining prediction
explanation = explainer.explain_instance(
    image.astype('double'),
    classifier_fn=lambda x: model.predict(tf.convert_to_tensor(x)),
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)

plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig("/kaggle/working/lime_explanation.png")
plt.close()


In [ ]:
import os
import pandas as pd
from collections import Counter

def get_class_distribution(directory):
    return Counter([folder for folder in os.listdir(directory) 
                    if os.path.isdir(os.path.join(directory, folder))])

train_dist = get_class_distribution(train_dir)
df = pd.DataFrame.from_dict(train_dist, orient='index', columns=['count'])

df.plot(kind='barh', figsize=(10, 8), color='green')
plt.title("Class Distribution - Train Set", fontsize=14)
plt.xlabel("Image Count")
plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution_train.png")
plt.close()


In [ ]:
import cv2
import matplotlib.cm as cm

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Select sample image from test set
for image, label in test_dataset.unbatch().take(1):
    img = tf.expand_dims(image, axis=0)
    heatmap = make_gradcam_heatmap(img, model, last_conv_layer_name='conv2d_5')  # Update to your last conv layer name

    heatmap = cv2.resize(heatmap, (img_size, img_size))
    heatmap_colored = cm.jet(heatmap)[:, :, :3]
    overlay = heatmap_colored * 0.4 + image.numpy() / 255.0
    plt.imshow(overlay)
    plt.title("Grad-CAM Overlay")
    plt.axis('off')
    plt.savefig('/kaggle/working/gradcam.png')
    plt.show()


In [ ]:
from sklearn.manifold import TSNE

features = []
labels = []

feature_extractor = Model(inputs=model.input, outputs=model.get_layer(index=-3).output)

for images, lbls in test_dataset:
    feats = feature_extractor.predict(images)
    features.append(feats)
    labels.extend(tf.argmax(lbls, axis=1).numpy())

features = np.concatenate(features, axis=0)
labels = np.array(labels)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_results = tsne.fit_transform(features)

plt.figure(figsize=(10, 8))
for i in range(num_classes):
    idxs = labels == i
    plt.scatter(tsne_results[idxs, 0], tsne_results[idxs, 1], label=class_names[i], s=15)
plt.legend(fontsize=9)
plt.title("t-SNE Visualization of Leaf Embeddings", fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/tsne_plot.png')
plt.show()


# Save Model for future

In [ ]:
model.save("/kaggle/working/medicinal_plant_classifier.h5")


In [ ]:
from tensorflow.keras.models import load_model

# If saved as .h5
model = load_model("/kaggle/working/medicinal_plant_classifier.h5")

# Or if saved in TensorFlow SavedModel format
# model = load_model("/kaggle/working/medicinal_plant_classifier")


# 2nd phase start form here

In [ ]:
# ============================================
# CLEAN FULL PIPELINE (QUIET LOGS) + MERGED FIGURE (300 DPI) + OPTIONAL t-SNE
# ============================================

# ---- Must be FIRST (before importing TF) ----
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress TF C++ logs (most of the noise)
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0" # optional: reduce extra backend logs

import json
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from IPython.display import display, Markdown

# Optional t-SNE (set ENABLE_TSNE=True)
from sklearn.manifold import TSNE

# -------------------------------
# PATHS
# -------------------------------
MODEL_PATH = "/kaggle/input/llm-rag-medinet-xg-work-json-csv-h5/medicinal_plant_classifier.h5"
JSON_PATH  = "/kaggle/input/llm-rag-medinet-xg-work-json-csv-h5/leaf_data.json"

# -------------------------------
# SETTINGS
# -------------------------------
ENABLE_TSNE = True           # set False if you don't need t-SNE
TSNE_MAX_PER_CLASS = 30      # how many images per class to sample (speed control)
TSNE_PERPLEXITY = 30         # typical 5-50
TSNE_RANDOM_STATE = 42
SAVE_DIR = "./outputs"
SAVE_PDF = True              # also save vector pdf
TOP_K = 5

# -------------------------------
# QUIET PYTHON/TF LOGS
# -------------------------------
tf.get_logger().setLevel("ERROR")
try:
    tf.autograph.set_verbosity(0)
except Exception:
    pass

# -------------------------------
# LOAD JSON + MODEL (quiet inference; avoids "compiled metrics" warning)
# -------------------------------
with open(JSON_PATH, "r", encoding="utf-8") as f:
    plant_data = json.load(f)

model = load_model(MODEL_PATH, compile=False)  # compile=False removes metrics warning
class_keys = sorted(list(plant_data.keys()))

# -------------------------------
# IMAGE PREPROCESSING
# -------------------------------
def load_and_preprocess_image(img_path, target_size=(224, 224)):
    img = image.load_img(img_path, target_size=target_size)
    arr = image.img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)

# -------------------------------
# PREDICT (Top-K)
# -------------------------------
def predict_plant_class(img_path, top_k=5):
    x = load_and_preprocess_image(img_path)
    preds = model.predict(x, verbose=0)[0]
    top_idx = preds.argsort()[-top_k:][::-1]
    top = [(class_keys[i], float(preds[i]) * 100.0) for i in top_idx]
    return top[0][0], top[0][1], top, x, preds

# -------------------------------
# JSON EXPLANATION + QUERY ANSWER
# -------------------------------
def generate_human_like_explanation(plant_key):
    info = plant_data.get(plant_key, {})
    if not info:
        return "No information available."

    text = f"{info.get('Scientific Name','Unknown')} ({plant_key}) is a {info.get('Life Span','')} {info.get('Plant Habit','')} in the {info.get('Botanical Family','')} family. "
    text += f"It naturally grows in {info.get('Native Habitat','')} across {info.get('Geographical Origin','')}. "
    text += f"Traditionally, it is used for {info.get('Traditional Uses','')} and contains active compounds like {info.get('Primary Active Compounds','')}. "
    text += f"Preferred cultivation conditions include {info.get('Climate Requirements','')} and {info.get('Preferred Soil Type','')}. "
    text += f"Safety considerations: {info.get('Pregnancy & Lactation Safety','')}. "
    text += f"Leaf features: {info.get('Leaf Morphology','')}. "
    return text

def answer_query(plant_key, query):
    info = plant_data.get(plant_key, {})
    if not info:
        return "No plant information found."

    q = query.lower()
    keywords = {
        "scientific": "Scientific Name",
        "botanical": "Botanical Family",
        "genus": "Genus & Species",
        "family": "Botanical Family",
        "leaf": "Leaf Morphology",
        "life": "Life Span",
        "habitat": "Native Habitat",
        "origin": "Geographical Origin",
        "uses": "Traditional Uses",
        "medicinal": "Therapeutic Properties",
        "dosage": "Standard Dosage",
        "toxic": "Toxicity Levels",
        "safety": "Pregnancy & Lactation Safety",
        "interaction": "Drug-Herb Interactions",
        "country": "Geographical Origin",
        "soil": "Preferred Soil Type",
        "climate": "Climate Requirements"
    }
    for key, json_key in keywords.items():
        if key in q:
            return info.get(json_key, "No information available for this query.")
    return "I cannot answer this question because it is outside my knowledge base."

# -------------------------------
# GRAD-CAM
# -------------------------------
def grad_cam(model, img_array, class_idx=None, last_conv_layer_name=None):
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if "conv" in layer.name:
                last_conv_layer_name = layer.name
                break
    if last_conv_layer_name is None:
        raise ValueError("No convolutional layer found in model.")

    last_conv_layer = model.get_layer(last_conv_layer_name)
    grad_model = tf.keras.models.Model([model.inputs], [last_conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if class_idx is None:
            class_idx = int(np.argmax(predictions[0]))
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap, 0) / (np.max(heatmap) + 1e-8)
    return heatmap

def overlay_heatmap(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, colormap)
    return cv2.addWeighted(img, 1 - alpha, heatmap_color, alpha, 0)

# -------------------------------
# REPORT TEXT (bottom panel)
# -------------------------------
def build_analysis_report(predicted_class, confidence, user_query=None, query_answer=None):
    lines = [
        "--- Analysis Report ---",
        f"Result: {predicted_class}",
        f"Confidence: {confidence:.2f}%",
        "Explanation: Model focused on highlighted regions (red/yellow).",
        "If highlights are on leaf edges/veins, prediction uses morphology.",
        "If highlights are mostly background, prediction may be unreliable."
    ]
    if user_query:
        lines += ["", f"Query: {user_query}", f"Answer: {query_answer if query_answer else 'N/A'}"]
    return "\n".join(lines)

# -------------------------------
# t-SNE SUPPORT (features from penultimate layer)
# -------------------------------
def build_embedding_model(base_model):
    # Pick a good embedding layer automatically:
    # prefer GlobalAveragePooling, otherwise last layer before final Dense.
    layer_name = None
    for layer in reversed(base_model.layers):
        name = layer.name.lower()
        if "global_average_pooling" in name or "gap" == name:
            layer_name = layer.name
            break
    if layer_name is None:
        # fallback: take second-to-last layer output
        layer_name = base_model.layers[-2].name

    emb_model = tf.keras.Model(inputs=base_model.input, outputs=base_model.get_layer(layer_name).output)
    return emb_model, layer_name

def get_embedding(embed_model, img_path):
    x = load_and_preprocess_image(img_path)
    v = embed_model.predict(x, verbose=0)
    return v.squeeze()

def collect_tsne_samples_from_kaggle_dataset(root_dir, max_per_class=30, exts=(".jpg", ".jpeg", ".png")):
    """
    root_dir should point to a folder where subfolders are class names (like Kaggle test/ or train/).
    Returns list of (img_path, class_label)
    """
    samples = []
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"t-SNE root_dir not found: {root_dir}")

    for cls in sorted(os.listdir(root_dir)):
        cls_path = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        count = 0
        for fn in os.listdir(cls_path):
            if fn.lower().endswith(exts):
                samples.append((os.path.join(cls_path, fn), cls))
                count += 1
                if count >= max_per_class:
                    break
    return samples

def compute_tsne_map(samples, embed_model, perplexity=30, random_state=42):
    X = np.array([get_embedding(embed_model, p) for p, _ in samples])
    y = np.array([lab for _, lab in samples])

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=random_state
    )
    Z = tsne.fit_transform(X)
    return Z, y

def plot_tsne(Z, y, title="t-SNE of Feature Embeddings", save_path=None, dpi=300):
    plt.figure(figsize=(9, 7), dpi=dpi)
    for cls in np.unique(y):
        idx = (y == cls)
        plt.scatter(Z[idx, 0], Z[idx, 1], s=10, label=cls)
    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
    plt.title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.show()

# -------------------------------
# MERGED FIGURE PIPELINE (+ optional t-SNE generation)
# -------------------------------
def process_plant_image_interactive_merged(top_k=5, save_dir="./outputs", save_pdf=True, enable_tsne=True):
    img_path = input("Enter the path to your plant image: ").strip()
    user_query = input("Enter your query about the plant (optional): ").strip()

    predicted_class, confidence, top_classes, img_array, _ = predict_plant_class(img_path, top_k=top_k)
    human_expl = generate_human_like_explanation(predicted_class)
    query_answer = answer_query(predicted_class, user_query) if user_query else ""

    heatmap = grad_cam(model, img_array)
    overlay_img = overlay_heatmap(img_path, heatmap)

    # Load original for panel 1
    orig_bgr = cv2.imread(img_path)
    if orig_bgr is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)

    # bar chart data
    labels = [c for c, _ in top_classes][::-1]
    scores = [p for _, p in top_classes][::-1]

    # Merged figure (publication-ready)
    fig = plt.figure(figsize=(16, 6), dpi=300)
    gs = fig.add_gridspec(2, 3, height_ratios=[4, 1], wspace=0.35, hspace=0.15)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(orig_rgb)
    ax1.set_title(f"Input Image\n({os.path.basename(img_path)})", fontsize=12)
    ax1.axis("off")

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(overlay_img)
    ax2.set_title("Visual Explanation\n(Grad-CAM Regions)", fontsize=12)
    ax2.axis("off")

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.barh(labels, scores)
    ax3.set_xlim(0, 100)
    ax3.set_xlabel("Confidence Score (%)")
    ax3.set_title(f"Top {top_k} Likely Classes", fontsize=12)
    for i, v in enumerate(scores):
        ax3.text(min(v + 1, 99), i, f"{v:.2f}%", va="center", fontsize=9)

    ax4 = fig.add_subplot(gs[1, :])
    ax4.axis("off")
    report_text = build_analysis_report(
        predicted_class,
        confidence,
        user_query if user_query else None,
        query_answer if user_query else None
    )
    ax4.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax4.transAxes, color="black", alpha=0.85))
    ax4.text(0.01, 0.5, report_text, transform=ax4.transAxes,
             fontsize=10, color="white", va="center", family="monospace")

    # Notebook text (like your example screenshot)
    display(Markdown(f"## ✅ Predicted Plant Class: **{predicted_class}**"))
    display(Markdown(f"**Prediction Confidence:** {confidence:.2f}%"))
    display(Markdown("### 🔹 Top Predictions:"))
    for c, p in top_classes:
        display(Markdown(f"- **{c}**: {p:.2f}%"))
    display(Markdown("### 🔹 Human-like Plant Explanation"))
    display(Markdown(human_expl))
    if user_query:
        display(Markdown("### 💡 Query Answer"))
        display(Markdown(query_answer))

    # Save merged figure
    os.makedirs(save_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(img_path))[0]
    out_png = os.path.join(save_dir, f"{base}_merged_gradcam_top{top_k}_300dpi.png")
    fig.savefig(out_png, dpi=300, bbox_inches="tight")

    out_pdf = None
    if save_pdf:
        out_pdf = os.path.join(save_dir, f"{base}_merged_gradcam_top{top_k}.pdf")
        fig.savefig(out_pdf, bbox_inches="tight")

    plt.show()
    display(Markdown(f"✅ Saved merged figure: `{out_png}`" + (f" and `{out_pdf}`" if out_pdf else "")))

    # -----------------------
    # OPTIONAL: t-SNE (MARK current input image point) + save 300 DPI
    # -----------------------
    if enable_tsne:
        # CHANGE THIS to your dataset root (folder contains class subfolders)
        TSNE_ROOT = input("Enter dataset root folder for t-SNE (e.g., .../test): ").strip()

        embed_model, emb_layer = build_embedding_model(model)
        display(Markdown(f"### 🧬 t-SNE Setup"))
        display(Markdown(f"- Embedding layer: `{emb_layer}`"))
        display(Markdown(f"- Sampling up to `{TSNE_MAX_PER_CLASS}` images per class"))

        # 1) collect dataset samples
        samples = collect_tsne_samples_from_kaggle_dataset(
            TSNE_ROOT, max_per_class=TSNE_MAX_PER_CLASS
        )

        # 2) compute dataset embeddings
        X_data = np.array([get_embedding(embed_model, p) for p, _ in samples])
        y_data = np.array([lab for _, lab in samples])

        # 3) compute query image embedding
        x_query = get_embedding(embed_model, img_path)
        X_all = np.vstack([X_data, x_query.reshape(1, -1)])
        y_all = np.append(y_data, "__QUERY_IMAGE__")  # special label

        # 4) fit t-SNE on (dataset + query)
        tsne = TSNE(
            n_components=2,
            perplexity=TSNE_PERPLEXITY,
            init="pca",
            learning_rate="auto",
            random_state=TSNE_RANDOM_STATE
        )
        Z_all = tsne.fit_transform(X_all)

        Z_data = Z_all[:-1]
        Z_query = Z_all[-1]

        # 5) plot and highlight query point
        tsne_png = os.path.join(save_dir, f"tsne_with_query_{os.path.basename(TSNE_ROOT)}_300dpi.png")

        plt.figure(figsize=(9, 7), dpi=300)

        # plot dataset points grouped by label
        for cls in np.unique(y_data):
            idx = (y_data == cls)
            plt.scatter(Z_data[idx, 0], Z_data[idx, 1], s=10, label=cls)

        # highlight query image point (big star + outline)
        plt.scatter(Z_query[0], Z_query[1], s=260, marker="*", edgecolors="black", linewidths=1.5)
        plt.annotate(
            f"QUERY\n({predicted_class})",
            (Z_query[0], Z_query[1]),
            textcoords="offset points",
            xytext=(10, 10),
            fontsize=10,
            fontweight="bold"
        )

        plt.title("t-SNE of Feature Embeddings (Query Highlighted)")
        plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
        plt.tight_layout()
        plt.savefig(tsne_png, dpi=300, bbox_inches="tight")
        plt.show()

        display(Markdown(f"✅ Saved t-SNE plot (query highlighted): `{tsne_png}`"))


# -------------------------------
# RUN
# -------------------------------
process_plant_image_interactive_merged(
    top_k=TOP_K,
    save_dir=SAVE_DIR,
    save_pdf=SAVE_PDF,
    enable_tsne=ENABLE_TSNE
)


# EVALUATION MODULE: Faithfulness + Alignment + Dashboard

In [ ]:
# ============================================
# EVALUATION MODULE: Faithfulness + Alignment + Dashboard
# ============================================

import re
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from difflib import SequenceMatcher

# -------------------------------
# Text utils
# -------------------------------
def _norm_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _tokenize(s: str):
    s = _norm_text(s)
    # keep words and numbers
    return re.findall(r"[a-z0-9]+", s)

def split_sentences(text: str):
    """Lightweight sentence splitter (works well enough for your generated paragraphs)."""
    if not text or not text.strip():
        return []
    # split on ., ?, !, newline
    parts = re.split(r"(?<=[\.\?\!])\s+|\n+", text.strip())
    sents = [p.strip() for p in parts if p.strip()]
    return sents

def seq_sim(a: str, b: str) -> float:
    """Fuzzy similarity using SequenceMatcher [0..1]."""
    a = _norm_text(a)
    b = _norm_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def jaccard(a_tokens, b_tokens) -> float:
    A, B = set(a_tokens), set(b_tokens)
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

# -------------------------------
# Build JSON evidence index
# -------------------------------
DEFAULT_EVIDENCE_FIELDS = [
    "Scientific Name",
    "Botanical Family",
    "Genus & Species",
    "Local/Vernacular Names",
    "Leaf Morphology",
    "Plant Habit",
    "Life Span",
    "Geographical Origin",
    "Native Habitat",
    "Global Distribution",
    "Preferred Soil Type",
    "Climate Requirements",
    "Primary Active Compounds",
    "Secondary Metabolites",
    "Essential Oils",
    "Therapeutic Properties",
    "Traditional Uses",
    "Modern Clinical Uses",
    "Medicinal Part Used",
    "Standard Dosage",
    "Internal Preparation Method",
    "External Application Method",
    "Extraction Process",
    "Culinary Uses",
    "Toxicity Levels",
    "Contraindications",
    "Pregnancy & Lactation Safety",
    "Drug-Herb Interactions",
    "Conservation Status",
    "Nutritional Profile",
]

def build_evidence_index(plant_info: dict, evidence_fields=None):
    """
    Returns:
      evidence_map: {field: normalized_string_value}
      token_map:    {field: token_list}
    """
    if evidence_fields is None:
        evidence_fields = DEFAULT_EVIDENCE_FIELDS

    evidence_map = {}
    token_map = {}
    for f in evidence_fields:
        val = plant_info.get(f, "")
        val_norm = _norm_text(val)
        evidence_map[f] = val_norm
        token_map[f] = _tokenize(val_norm)
    return evidence_map, token_map

# -------------------------------
# Claim alignment + faithfulness scoring
# -------------------------------
def align_sentence_to_json(sentence: str, evidence_map: dict, token_map: dict,
                           min_seq_sim=0.58, min_token_jacc=0.08, min_overlap_tokens=2):
    """
    Determine which JSON fields support a sentence.

    Returns:
      supports: list of dicts [{field, seq_sim, token_jacc, overlap_tokens}]
      supported: bool
      best_score: float (0..1)
    """
    sent_norm = _norm_text(sentence)
    sent_tokens = _tokenize(sent_norm)
    if not sent_tokens:
        return [], True, 1.0  # empty/degenerate sentence doesn't count against faithfulness

    supports = []
    best = 0.0

    for field, ev in evidence_map.items():
        if not ev:
            continue

        # similarity modes:
        ssim = seq_sim(sent_norm, ev)

        # token overlap (robust to word order)
        ev_tokens = token_map.get(field, [])
        jac = jaccard(sent_tokens, ev_tokens)

        overlap = len(set(sent_tokens) & set(ev_tokens))

        # accept a field as "support" if it passes either fuzzy similarity
        # or a minimal token overlap threshold (useful for long values)
        ok = (ssim >= min_seq_sim) or ((jac >= min_token_jacc) and (overlap >= min_overlap_tokens))

        score = max(ssim, jac)
        best = max(best, score)

        if ok:
            supports.append({
                "field": field,
                "seq_sim": float(ssim),
                "token_jacc": float(jac),
                "overlap_tokens": int(overlap),
                "score": float(score),
            })

    supports = sorted(supports, key=lambda d: d["score"], reverse=True)

    supported = len(supports) > 0
    return supports, supported, float(best)

def faithfulness_score(text: str, plant_info: dict,
                       evidence_fields=None,
                       min_seq_sim=0.58, min_token_jacc=0.08, min_overlap_tokens=2):
    """
    Sentence-level faithfulness:
      faithfulness = supported_sentences / total_sentences

    Returns:
      report dict with sentence details + metrics
    """
    if evidence_fields is None:
        evidence_fields = DEFAULT_EVIDENCE_FIELDS

    evidence_map, token_map = build_evidence_index(plant_info, evidence_fields)

    sents = split_sentences(text)
    if not sents:
        return {
            "faithfulness": 0.0,
            "supported_sentences": 0,
            "total_sentences": 0,
            "unsupported_sentences": 0,
            "sentences": [],
            "coverage": 0.0,
            "covered_fields": [],
        }

    sentence_rows = []
    supported_count = 0
    covered_fields = set()

    for i, s in enumerate(sents, start=1):
        supports, supported, best = align_sentence_to_json(
            s, evidence_map, token_map,
            min_seq_sim=min_seq_sim,
            min_token_jacc=min_token_jacc,
            min_overlap_tokens=min_overlap_tokens
        )
        if supported:
            supported_count += 1
            for sp in supports[:3]:
                covered_fields.add(sp["field"])

        sentence_rows.append({
            "sent_id": i,
            "sentence": s,
            "supported": supported,
            "best_score": best,
            "top_support_fields": [sp["field"] for sp in supports[:3]],
            "top_support_scores": [round(sp["score"], 3) for sp in supports[:3]],
        })

    total = len(sents)
    unsupported = total - supported_count

    # coverage: how many evidence fields were mentioned/supported at least once
    coverage = len(covered_fields) / max(1, len(evidence_fields))

    return {
        "faithfulness": supported_count / total,
        "supported_sentences": supported_count,
        "total_sentences": total,
        "unsupported_sentences": unsupported,
        "sentences": sentence_rows,
        "coverage": coverage,
        "covered_fields": sorted(list(covered_fields)),
    }

# -------------------------------
# Batch evaluation + dashboard
# -------------------------------
def evaluate_explanations_batch(plant_data: dict, generator_fn,
                                out_dir="./outputs/eval",
                                evidence_fields=None,
                                min_seq_sim=0.58, min_token_jacc=0.08, min_overlap_tokens=2,
                                limit=None):
    """
    Evaluates generated explanations for many plant keys.

    generator_fn: function(plant_key)->text
      e.g., your generate_human_like_explanation

    Outputs:
      - explanations_eval_summary.csv (per plant)
      - explanations_eval_sentences.csv (per sentence)
      - plots (png): faithfulness hist, coverage hist, unsupported rate

    Returns:
      summary_df, sentences_df
    """
    os.makedirs(out_dir, exist_ok=True)
    if evidence_fields is None:
        evidence_fields = DEFAULT_EVIDENCE_FIELDS

    keys = list(plant_data.keys())
    keys = sorted(keys)
    if limit is not None:
        keys = keys[:int(limit)]

    summary_rows = []
    sentence_rows = []

    for plant_key in keys:
        info = plant_data[plant_key]
        text = generator_fn(plant_key)

        rep = faithfulness_score(
            text, info,
            evidence_fields=evidence_fields,
            min_seq_sim=min_seq_sim,
            min_token_jacc=min_token_jacc,
            min_overlap_tokens=min_overlap_tokens
        )

        summary_rows.append({
            "plant_key": plant_key,
            "scientific_name": info.get("Scientific Name", ""),
            "faithfulness": rep["faithfulness"],
            "coverage": rep["coverage"],
            "supported_sentences": rep["supported_sentences"],
            "unsupported_sentences": rep["unsupported_sentences"],
            "total_sentences": rep["total_sentences"],
            "covered_fields_count": len(rep["covered_fields"]),
        })

        for srow in rep["sentences"]:
            sentence_rows.append({
                "plant_key": plant_key,
                "scientific_name": info.get("Scientific Name", ""),
                **srow
            })

    summary_df = pd.DataFrame(summary_rows)
    sentences_df = pd.DataFrame(sentence_rows)

    # Save CSVs
    summary_csv = os.path.join(out_dir, "explanations_eval_summary.csv")
    sent_csv = os.path.join(out_dir, "explanations_eval_sentences.csv")
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8")
    sentences_df.to_csv(sent_csv, index=False, encoding="utf-8")

    # Plots
    # 1) faithfulness distribution
    plt.figure(figsize=(8, 5), dpi=200)
    plt.hist(summary_df["faithfulness"], bins=10)
    plt.title("Faithfulness Distribution (sentence-supported ratio)")
    plt.xlabel("Faithfulness")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "faithfulness_hist.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # 2) coverage distribution
    plt.figure(figsize=(8, 5), dpi=200)
    plt.hist(summary_df["coverage"], bins=10)
    plt.title("Coverage Distribution (supported fields / total fields)")
    plt.xlabel("Coverage")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "coverage_hist.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # 3) unsupported sentence rate
    rate = summary_df["unsupported_sentences"] / summary_df["total_sentences"].replace(0, np.nan)
    plt.figure(figsize=(8, 5), dpi=200)
    plt.hist(rate.dropna(), bins=10)
    plt.title("Unsupported Sentence Rate Distribution")
    plt.xlabel("Unsupported rate")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "unsupported_rate_hist.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # Identify top problematic plants (lowest faithfulness)
    worst = summary_df.sort_values(["faithfulness", "coverage"], ascending=True).head(10)
    worst_csv = os.path.join(out_dir, "worst_10_plants.csv")
    worst.to_csv(worst_csv, index=False, encoding="utf-8")

    print("✅ Saved evaluation outputs:")
    print(" -", summary_csv)
    print(" -", sent_csv)
    print(" -", os.path.join(out_dir, "faithfulness_hist.png"))
    print(" -", os.path.join(out_dir, "coverage_hist.png"))
    print(" -", os.path.join(out_dir, "unsupported_rate_hist.png"))
    print(" -", worst_csv)

    return summary_df, sentences_df

# -------------------------------
# Single-sample alignment viewer (human inspection)
# -------------------------------
def show_alignment_report(plant_key: str, generated_text: str, plant_data: dict,
                          evidence_fields=None,
                          min_seq_sim=0.58, min_token_jacc=0.08, min_overlap_tokens=2,
                          max_supports_to_show=3):
    """
    Prints a readable alignment report for one explanation.
    """
    if evidence_fields is None:
        evidence_fields = DEFAULT_EVIDENCE_FIELDS

    info = plant_data.get(plant_key, {})
    if not info:
        print("Plant key not found.")
        return

    evidence_map, token_map = build_evidence_index(info, evidence_fields)
    sents = split_sentences(generated_text)

    print("="*80)
    print(f"PLANT: {plant_key} | {info.get('Scientific Name','')}")
    print("="*80)
    print("Generated text:\n", generated_text)
    print("-"*80)

    unsupported = 0
    for i, s in enumerate(sents, start=1):
        supports, supported, best = align_sentence_to_json(
            s, evidence_map, token_map,
            min_seq_sim=min_seq_sim,
            min_token_jacc=min_token_jacc,
            min_overlap_tokens=min_overlap_tokens
        )
        flag = "✅ SUPPORTED" if supported else "❌ UNSUPPORTED"
        if not supported:
            unsupported += 1

        print(f"[{i}] {flag} | best_score={best:.3f}")
        print("  Sentence:", s)

        if supports:
            print("  Top supports:")
            for sp in supports[:max_supports_to_show]:
                ev_preview = info.get(sp["field"], "")
                ev_preview = (ev_preview[:120] + "...") if len(ev_preview) > 120 else ev_preview
                print(f"   - {sp['field']} (score={sp['score']:.3f}, overlap={sp['overlap_tokens']}): {ev_preview}")
        else:
            print("  Top supports: None")

        print("-"*80)

    print(f"Unsupported sentences: {unsupported}/{len(sents)}")
    print("="*80)


In [ ]:
pk = "1_1_thankuni"  # pick any key
text = generate_human_like_explanation(pk)

show_alignment_report(
    plant_key=pk,
    generated_text=text,
    plant_data=plant_data
)


# Evaluation Cell

In [ ]:
# ============================================================
# - Classification: Confusion Matrix + Macro Precision/Recall/F1 + Top-k Acc
# - QA (JSON-grounded): Exact Match + Token-F1 + Field Confusion Matrix
# - Explanation (grounded NLG): Faithfulness + Coverage + Hallucination rate
# - Saves CSV + high-DPI plots to ./outputs/eval
# ============================================================

import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_recall_fscore_support, top_k_accuracy_score
)

# -----------------------------
# REQUIRED: you already have these in your notebook:
# - model (Keras model)
# - plant_data (loaded JSON dict)
# - class_keys (sorted list(plant_data.keys()))
# - load_and_preprocess_image(img_path, target_size=(224,224))
# - answer_query(plant_key, query)
# - generate_human_like_explanation(plant_key)
# -----------------------------

# ============ CONFIG ============
TEST_ROOT = "/kaggle/input/bangladeshi-weedy-area-medicinal-plant/medicinal_weedy_area_image_dataset_version2/medicinal_weedy_area_image_dataset_version2/test"  # <-- CHANGE: folder with class subfolders
OUT_DIR   = "./outputs/eval"
TOP_K     = 5
DPI       = 300

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def _norm(s):
    s = "" if s is None else str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def _tokenize(s):
    return re.findall(r"[a-z0-9]+", _norm(s))

def token_f1(pred, gt):
    p = _tokenize(pred); g = _tokenize(gt)
    if len(p)==0 and len(g)==0: return 1.0
    if len(p)==0 or len(g)==0: return 0.0
    P, G = set(p), set(g)
    inter = len(P & G)
    prec = inter / max(1, len(P))
    rec  = inter / max(1, len(G))
    if prec + rec == 0: return 0.0
    return 2*prec*rec/(prec+rec)

def mean_ci95(x):
    x = np.array(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan, np.nan, (np.nan, np.nan)
    m = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
    se = sd / np.sqrt(len(x)) if len(x) > 1 else 0.0
    ci = (m - 1.96*se, m + 1.96*se)
    return m, sd, ci

# ============================================================
# 1) CLASSIFICATION EVALUATION (Confusion Matrix + Macro-F1 etc.)
# ============================================================
def collect_test_images(test_root, exts=(".jpg",".jpeg",".png")):
    rows = []
    for cls in sorted(os.listdir(test_root)):
        cls_path = os.path.join(test_root, cls)
        if not os.path.isdir(cls_path):
            continue
        for fn in os.listdir(cls_path):
            if fn.lower().endswith(exts):
                rows.append({"img_path": os.path.join(cls_path, fn), "true_label": cls})
    return pd.DataFrame(rows)

def predict_probs(img_path):
    x = load_and_preprocess_image(img_path)
    probs = model.predict(x, verbose=0)[0]
    return probs

def eval_classifier(test_root, class_keys, top_k=5):
    df = collect_test_images(test_root)
    if df.empty:
        raise FileNotFoundError(f"No images found under: {test_root}")

    # Ensure labels in dataset match your JSON keys
    unknown = sorted(set(df["true_label"]) - set(class_keys))
    if unknown:
        print("⚠️ These folders are not in class_keys (JSON keys). Fix naming or mapping:")
        print(unknown[:20], ("..." if len(unknown) > 20 else ""))

    y_true = []
    y_pred = []
    prob_rows = []

    for _, r in df.iterrows():
        probs = predict_probs(r["img_path"])
        pred_idx = int(np.argmax(probs))
        pred_lab = class_keys[pred_idx]

        y_true.append(r["true_label"])
        y_pred.append(pred_lab)
        prob_rows.append(probs)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    prob_rows = np.vstack(prob_rows)

    # Core metrics
    acc = accuracy_score(y_true, y_pred)

    # Macro P/R/F1
    labels = [c for c in class_keys if c in set(y_true)]
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )

    # Top-k accuracy (needs one-hot)
    # sklearn expects y_true as indices for top_k_accuracy_score
    true_idx = np.array([class_keys.index(t) if t in class_keys else -1 for t in y_true])
    valid_mask = true_idx >= 0
    topk = top_k_accuracy_score(true_idx[valid_mask], prob_rows[valid_mask], k=top_k, labels=np.arange(len(class_keys)))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Save: classification report
    rep = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_df.to_csv(os.path.join(OUT_DIR, "classification_report.csv"), index=True)

    # Save predictions
    pred_df = pd.DataFrame({
        "img_path": df["img_path"].values,
        "true_label": y_true,
        "pred_label": y_pred
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "classification_predictions.csv"), index=False)

    # Plot confusion matrix (journal-friendly: high DPI, readable)
    plt.figure(figsize=(10, 8), dpi=DPI)
    plt.imshow(cm, aspect="auto")
    plt.title("Confusion Matrix (Test Set)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.xticks(ticks=np.arange(len(labels)), labels=labels, rotation=90, fontsize=6)
    plt.yticks(ticks=np.arange(len(labels)), labels=labels, fontsize=6)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=DPI, bbox_inches="tight")
    plt.show()

    summary = {
        "Accuracy": acc,
        "Macro-Precision": float(p),
        "Macro-Recall": float(r),
        "Macro-F1": float(f1),
        f"Top-{top_k}-Accuracy": float(topk),
        "N_images": int(len(y_true)),
        "N_classes_in_test": int(len(labels)),
    }
    pd.DataFrame([summary]).to_csv(os.path.join(OUT_DIR, "classification_summary.csv"), index=False)
    return summary, cm, labels

cls_summary, cm, cm_labels = eval_classifier(TEST_ROOT, class_keys, top_k=TOP_K)
print("✅ Classification summary:", cls_summary)
print(f"✅ Saved to: {OUT_DIR}/classification_* and confusion_matrix.png")


# ============================================================
# 2) QA (JSON-GROUNDED) EVALUATION: EM + Token-F1 + Field Confusion Matrix
# ============================================================
# We create a standard query set (journal-style: fixed, reproducible prompts).
QUERY_SPECS = [
    ("scientific name", "Scientific Name"),
    ("botanical family", "Botanical Family"),
    ("genus and species", "Genus & Species"),
    ("leaf morphology", "Leaf Morphology"),
    ("life span", "Life Span"),
    ("native habitat", "Native Habitat"),
    ("geographical origin", "Geographical Origin"),
    ("traditional uses", "Traditional Uses"),
    ("therapeutic properties", "Therapeutic Properties"),
    ("standard dosage", "Standard Dosage"),
    ("toxicity levels", "Toxicity Levels"),
    ("pregnancy & lactation safety", "Pregnancy & Lactation Safety"),
    ("drug-herb interactions", "Drug-Herb Interactions"),
    ("preferred soil type", "Preferred Soil Type"),
    ("climate requirements", "Climate Requirements"),
]

# Reverse map: if your answer_query() is keyword-driven, we can infer the "field chosen"
# by checking which key substring is present in query; this gives a field-level confusion matrix.
KEYWORD_TO_FIELD = {
    "scientific": "Scientific Name",
    "botanical": "Botanical Family",
    "genus": "Genus & Species",
    "family": "Botanical Family",
    "leaf": "Leaf Morphology",
    "life": "Life Span",
    "habitat": "Native Habitat",
    "origin": "Geographical Origin",
    "uses": "Traditional Uses",
    "medicinal": "Therapeutic Properties",
    "dosage": "Standard Dosage",
    "toxic": "Toxicity Levels",
    "safety": "Pregnancy & Lactation Safety",
    "interaction": "Drug-Herb Interactions",
    "country": "Geographical Origin",
    "soil": "Preferred Soil Type",
    "climate": "Climate Requirements",
}

def infer_field_from_query(q):
    qn = _norm(q)
    for k, f in KEYWORD_TO_FIELD.items():
        if k in qn:
            return f
    return "__UNKNOWN__"

def eval_qa(plant_data):
    rows = []
    conf_rows = []

    for plant_key, info in plant_data.items():
        for qtext, gt_field in QUERY_SPECS:
            pred_ans = answer_query(plant_key, qtext)
            gt_ans = info.get(gt_field, "")

            em = 1.0 if _norm(pred_ans) == _norm(gt_ans) else 0.0
            f1 = token_f1(pred_ans, gt_ans)

            chosen_field = infer_field_from_query(qtext)
            conf_rows.append({"gt_field": gt_field, "pred_field": chosen_field})

            rows.append({
                "plant_key": plant_key,
                "scientific_name": info.get("Scientific Name", ""),
                "query": qtext,
                "gt_field": gt_field,
                "pred_field": chosen_field,
                "exact_match": em,
                "token_f1": f1
            })

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT_DIR, "qa_eval_rows.csv"), index=False)

    em_mean = float(df["exact_match"].mean())
    f1_mean = float(df["token_f1"].mean())

    # Field confusion matrix
    conf_df = pd.DataFrame(conf_rows)
    fields = sorted(set(conf_df["gt_field"]) | set(conf_df["pred_field"]))
    fcm = pd.crosstab(conf_df["gt_field"], conf_df["pred_field"]).reindex(index=fields, columns=fields, fill_value=0)
    fcm.to_csv(os.path.join(OUT_DIR, "qa_field_confusion_matrix.csv"))

    # Plot field confusion matrix
    plt.figure(figsize=(10, 8), dpi=DPI)
    plt.imshow(fcm.values, aspect="auto")
    plt.title("Field Confusion Matrix (QA)")
    plt.xlabel("Predicted Field (by query mapping)")
    plt.ylabel("Ground Truth Field")
    plt.colorbar()
    plt.xticks(np.arange(len(fields)), fields, rotation=90, fontsize=6)
    plt.yticks(np.arange(len(fields)), fields, fontsize=6)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "qa_field_confusion_matrix.png"), dpi=DPI, bbox_inches="tight")
    plt.show()

    qa_summary = {"QA_ExactMatch_Mean": em_mean, "QA_TokenF1_Mean": f1_mean, "N_QA_samples": int(len(df))}
    pd.DataFrame([qa_summary]).to_csv(os.path.join(OUT_DIR, "qa_summary.csv"), index=False)
    return qa_summary, df, fcm

qa_summary, qa_rows_df, qa_field_cm = eval_qa(plant_data)
print("✅ QA summary:", qa_summary)
print(f"✅ Saved to: {OUT_DIR}/qa_*")


# ============================================================
# 3) EXPLANATION (GROUNDED NLG) EVALUATION: Faithfulness + Coverage + Hallucination
#    NOTE: This uses a simple, reproducible automatic checker.
# ============================================================
EVIDENCE_FIELDS = [
    "Scientific Name","Botanical Family","Genus & Species","Leaf Morphology","Plant Habit","Life Span",
    "Native Habitat","Geographical Origin","Traditional Uses","Primary Active Compounds",
    "Preferred Soil Type","Climate Requirements","Pregnancy & Lactation Safety","Toxicity Levels",
    "Drug-Herb Interactions"
]

def build_evidence_map(info, fields):
    return {f: _norm(info.get(f, "")) for f in fields}

def sentence_split(text):
    if not text: return []
    parts = re.split(r"(?<=[\.\?\!])\s+|\n+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def seq_ratio(a, b):
    # fast-ish, token overlap based proxy (keeps it dependency-free)
    A = set(_tokenize(a)); B = set(_tokenize(b))
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

def eval_explanations(plant_data, min_support=0.08):
    rows = []
    sent_rows = []
    for plant_key, info in plant_data.items():
        text = generate_human_like_explanation(plant_key)
        sents = sentence_split(text)
        ev = build_evidence_map(info, EVIDENCE_FIELDS)

        supported = 0
        covered = set()

        for i, s in enumerate(sents, 1):
            best_field = None
            best_score = 0.0
            for f, val in ev.items():
                if not val: 
                    continue
                sc = seq_ratio(s, val)
                if sc > best_score:
                    best_score = sc
                    best_field = f

            is_supported = best_score >= min_support
            if is_supported:
                supported += 1
                covered.add(best_field)

            sent_rows.append({
                "plant_key": plant_key,
                "sent_id": i,
                "sentence": s,
                "supported": is_supported,
                "best_field": best_field if best_field else "__NONE__",
                "best_score": float(best_score)
            })

        total = len(sents)
        faith = supported / total if total > 0 else np.nan
        cov = len(covered) / len(EVIDENCE_FIELDS)

        rows.append({
            "plant_key": plant_key,
            "scientific_name": info.get("Scientific Name",""),
            "faithfulness": float(faith) if not np.isnan(faith) else np.nan,
            "coverage": float(cov),
            "hallucination_rate": float(1-faith) if not np.isnan(faith) else np.nan,
            "supported_sentences": supported,
            "total_sentences": total
        })

    df = pd.DataFrame(rows)
    sdf = pd.DataFrame(sent_rows)

    df.to_csv(os.path.join(OUT_DIR, "explanations_summary.csv"), index=False)
    sdf.to_csv(os.path.join(OUT_DIR, "explanations_sentences.csv"), index=False)

    # Stats (mean ± std + 95% CI)
    f_m, f_sd, f_ci = mean_ci95(df["faithfulness"])
    c_m, c_sd, c_ci = mean_ci95(df["coverage"])
    h_m, h_sd, h_ci = mean_ci95(df["hallucination_rate"])

    stats = pd.DataFrame([{
        "Faithfulness_mean": f_m, "Faithfulness_sd": f_sd, "Faithfulness_CI95_low": f_ci[0], "Faithfulness_CI95_high": f_ci[1],
        "Coverage_mean": c_m, "Coverage_sd": c_sd, "Coverage_CI95_low": c_ci[0], "Coverage_CI95_high": c_ci[1],
        "Hallucination_mean": h_m, "Hallucination_sd": h_sd, "Hallucination_CI95_low": h_ci[0], "Hallucination_CI95_high": h_ci[1],
        "N_plants": int(len(df))
    }])
    stats.to_csv(os.path.join(OUT_DIR, "explanations_stats_ci95.csv"), index=False)

    # Plots
    for col, title, fname in [
        ("faithfulness", "Faithfulness Distribution", "faithfulness_hist.png"),
        ("coverage", "Coverage Distribution", "coverage_hist.png"),
        ("hallucination_rate", "Hallucination Rate Distribution", "hallucination_hist.png"),
    ]:
        plt.figure(figsize=(7, 4), dpi=DPI)
        plt.hist(df[col].dropna().values, bins=10)
        plt.title(title)
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, fname), dpi=DPI, bbox_inches="tight")
        plt.show()

    # What fields are most often used as "best_field" (helps paper discussion)
    field_counts = sdf[sdf["supported"] == True]["best_field"].value_counts()
    field_counts.to_csv(os.path.join(OUT_DIR, "explanations_top_support_fields.csv"))

    return stats, df, sdf

exp_stats, exp_df, exp_sent_df = eval_explanations(plant_data, min_support=0.08)
print("✅ Explanation (grounded NLG) stats (mean±sd + 95% CI):")
display(exp_stats)
print(f"✅ Saved to: {OUT_DIR}/explanations_*")


# ============================================================
# 4) FINAL "JOURNAL TABLE" (single CSV you can paste into paper)
# ============================================================
final = {
    "Classification_Accuracy": cls_summary["Accuracy"],
    "Classification_MacroF1": cls_summary["Macro-F1"],
    f"Classification_Top{TOP_K}_Accuracy": cls_summary[f"Top-{TOP_K}-Accuracy"],
    "QA_ExactMatch": qa_summary["QA_ExactMatch_Mean"],
    "QA_TokenF1": qa_summary["QA_TokenF1_Mean"],
    "Explanation_Faithfulness_mean": float(exp_stats["Faithfulness_mean"].iloc[0]),
    "Explanation_Faithfulness_sd": float(exp_stats["Faithfulness_sd"].iloc[0]),
    "Explanation_Faithfulness_CI95_low": float(exp_stats["Faithfulness_CI95_low"].iloc[0]),
    "Explanation_Faithfulness_CI95_high": float(exp_stats["Faithfulness_CI95_high"].iloc[0]),
    "Explanation_Coverage_mean": float(exp_stats["Coverage_mean"].iloc[0]),
    "Explanation_Coverage_sd": float(exp_stats["Coverage_sd"].iloc[0]),
    "Explanation_Hallucination_mean": float(exp_stats["Hallucination_mean"].iloc[0]),
}

final_df = pd.DataFrame([final])
final_df.to_csv(os.path.join(OUT_DIR, "journal_summary_table.csv"), index=False)
print("✅ Saved journal table:", os.path.join(OUT_DIR, "journal_summary_table.csv"))
display(final_df)


In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_recall_fscore_support, top_k_accuracy_score
)

# -----------------------------
# CONFIG
# -----------------------------
TEST_ROOT = "/kaggle/input/bangladeshi-weedy-area-medicinal-plant/medicinal_weedy_area_image_dataset_version2/medicinal_weedy_area_image_dataset_version2/test"
OUT_DIR   = "/kaggle/working/outputs/eval"
TOP_K     = 5
DPI       = 300
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Normalization for label matching (robust)
# - removes anything inside parentheses
# - keeps letters, numbers, underscores only
# - compresses multiple underscores
# -----------------------------
def normalize_label(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\([^)]*\)", "", s)              # remove "(...)" parts
    s = re.sub(r"[^a-z0-9_]+", "_", s)           # non allowed -> underscore
    s = re.sub(r"_+", "_", s).strip("_")         # collapse underscores
    return s

# -----------------------------
# Build mapping: folder_name -> json_key
# -----------------------------
json_keys = list(class_keys)  # from your loaded JSON keys
norm_json_map = {normalize_label(k): k for k in json_keys}  # normalized -> original

def map_folder_to_json_key(folder_label: str):
    n = normalize_label(folder_label)
    return norm_json_map.get(n, None)

# -----------------------------
# Collect test images + mapped labels
# -----------------------------
def collect_test_images_mapped(test_root, exts=(".jpg",".jpeg",".png")):
    rows = []
    unmapped_folders = []

    for folder in sorted(os.listdir(test_root)):
        folder_path = os.path.join(test_root, folder)
        if not os.path.isdir(folder_path):
            continue

        mapped = map_folder_to_json_key(folder)
        if mapped is None:
            unmapped_folders.append(folder)
            continue

        for fn in os.listdir(folder_path):
            if fn.lower().endswith(exts):
                rows.append({
                    "img_path": os.path.join(folder_path, fn),
                    "true_folder": folder,
                    "true_label": mapped
                })

    return pd.DataFrame(rows), unmapped_folders

def predict_probs(img_path):
    x = load_and_preprocess_image(img_path)
    return model.predict(x, verbose=0)[0]

def eval_classifier_with_mapping(test_root, class_keys, top_k=5):
    df, unmapped = collect_test_images_mapped(test_root)

    print("✅ Images collected (mapped):", len(df))
    if unmapped:
        print("⚠️ Unmapped folders (will be ignored):", unmapped)

    if df.empty:
        raise ValueError("No mapped images found. Mapping failed.")

    y_true = df["true_label"].values
    prob_rows = []
    y_pred = []

    for p in df["img_path"].values:
        probs = predict_probs(p)
        prob_rows.append(probs)
        y_pred.append(class_keys[int(np.argmax(probs))])

    prob_rows = np.vstack(prob_rows)
    y_pred = np.array(y_pred)

    # Metrics
    acc = accuracy_score(y_true, y_pred)
    labels = sorted(list(set(y_true)))  # only classes present in test

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )

    true_idx = np.array([class_keys.index(t) for t in y_true])
    topk = top_k_accuracy_score(true_idx, prob_rows, k=top_k, labels=np.arange(len(class_keys)))

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Save predictions
    pred_df = df.copy()
    pred_df["pred_label"] = y_pred
    pred_path = os.path.join(OUT_DIR, "classification_predictions.csv")
    pred_df.to_csv(pred_path, index=False)

    # Save report
    rep = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = os.path.join(OUT_DIR, "classification_report.csv")
    rep_df.to_csv(rep_path, index=True)

    # Save summary
    summary = {
        "Accuracy": float(acc),
        "Macro-Precision": float(p),
        "Macro-Recall": float(r),
        "Macro-F1": float(f1),
        f"Top-{top_k}-Accuracy": float(topk),
        "N_images": int(len(y_true)),
        "N_classes_in_test": int(len(labels)),
        "Unmapped_folders_count": int(len(unmapped))
    }
    summary_path = os.path.join(OUT_DIR, "classification_summary.csv")
    pd.DataFrame([summary]).to_csv(summary_path, index=False)

    # Plot confusion matrix
    plt.figure(figsize=(12, 10), dpi=DPI)
    plt.imshow(cm, aspect="auto")
    plt.title("Confusion Matrix (Mapped Test Labels)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.xticks(np.arange(len(labels)), labels, rotation=90, fontsize=6)
    plt.yticks(np.arange(len(labels)), labels, fontsize=6)
    plt.tight_layout()
    cm_path = os.path.join(OUT_DIR, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=DPI, bbox_inches="tight")
    plt.show()

    print("✅ Saved:", pred_path)
    print("✅ Saved:", rep_path)
    print("✅ Saved:", summary_path)
    print("✅ Saved:", cm_path)

    return summary

cls_summary = eval_classifier_with_mapping(TEST_ROOT, class_keys, top_k=TOP_K)
print("✅ Classification summary:", cls_summary)

print("\n📁 Files in OUT_DIR:")
for fn in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, fn)
    print(" -", fn, f"({os.path.getsize(fp)} bytes)")


# ANSWER GENERATION EVALUATION (JSON-GROUNDED)

In [ ]:
# ============================================================
# - Faithfulness (supported sentence ratio)
# - Hallucination rate
# - Coverage (required field coverage)
# - Alignment matrix (field attribution counts)
# - Saves CSV + plots to /kaggle/working/outputs/answer_eval
# ============================================================

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from difflib import SequenceMatcher

# -----------------------------
# REQUIRED (already in your notebook):
# plant_data : dict  (loaded leaf_data.json)
# generate_human_like_explanation(plant_key) : str
# -----------------------------

OUT_DIR = "/kaggle/working/outputs/answer_eval"
DPI = 300
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Choose which fields count as "ground truth evidence"
# (Use the same set in your paper for reproducibility)
# -----------------------------
EVIDENCE_FIELDS = [
    "Scientific Name", "Botanical Family", "Genus & Species",
    "Life Span", "Plant Habit", "Leaf Morphology",
    "Native Habitat", "Geographical Origin",
    "Traditional Uses", "Therapeutic Properties",
    "Primary Active Compounds",
    "Preferred Soil Type", "Climate Requirements",
    "Toxicity Levels", "Pregnancy & Lactation Safety",
    "Drug-Herb Interactions"
]

# -----------------------------
# Text utils
# -----------------------------
def norm(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def tokenize(s: str):
    return re.findall(r"[a-z0-9]+", norm(s))

def split_sentences(text: str):
    if not text or not str(text).strip():
        return []
    parts = re.split(r"(?<=[\.\!\?])\s+|\n+", str(text).strip())
    return [p.strip() for p in parts if p.strip()]

def seq_sim(a: str, b: str) -> float:
    a = norm(a); b = norm(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def jaccard(a_tokens, b_tokens) -> float:
    A, B = set(a_tokens), set(b_tokens)
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

def mean_sd_ci95(x):
    x = np.array(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan, np.nan, (np.nan, np.nan)
    m = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
    se = sd / np.sqrt(len(x)) if len(x) > 1 else 0.0
    ci = (m - 1.96*se, m + 1.96*se)
    return m, sd, ci

# -----------------------------
# Alignment + Support decision
# -----------------------------
def best_support_field(sentence, info, fields):
    """
    Returns best matching field + score for a sentence.
    Score is max(seq_sim, token_jaccard) to capture paraphrase + overlap.
    """
    s_tokens = tokenize(sentence)
    best_f, best_score = None, 0.0

    for f in fields:
        ev = info.get(f, "")
        if not ev: 
            continue
        ev_tokens = tokenize(ev)
        score = max(seq_sim(sentence, ev), jaccard(s_tokens, ev_tokens))
        if score > best_score:
            best_score = score
            best_f = f

    return best_f, float(best_score)

def evaluate_one_explanation(plant_key, info, text, fields, support_threshold=0.18):
    """
    support_threshold:
      0.18 is a reasonable journal-friendly starting point.
      Increase to be stricter, decrease to be more lenient.
    """
    sents = split_sentences(text)
    if len(sents) == 0:
        return {
            "plant_key": plant_key,
            "sentences": 0,
            "supported_sentences": 0,
            "faithfulness": np.nan,
            "hallucination_rate": np.nan,
            "coverage": 0.0,
            "covered_fields": [],
            "rows": []
        }

    rows = []
    supported = 0
    covered = set()

    for i, s in enumerate(sents, start=1):
        f, score = best_support_field(s, info, fields)
        is_supported = (f is not None) and (score >= support_threshold)
        if is_supported:
            supported += 1
            covered.add(f)

        rows.append({
            "plant_key": plant_key,
            "sent_id": i,
            "sentence": s,
            "best_field": f if f else "__NONE__",
            "best_score": score,
            "supported": bool(is_supported)
        })

    faith = supported / len(sents)
    hall = 1.0 - faith
    cov = len(covered) / len(fields)

    return {
        "plant_key": plant_key,
        "sentences": len(sents),
        "supported_sentences": supported,
        "faithfulness": float(faith),
        "hallucination_rate": float(hall),
        "coverage": float(cov),
        "covered_fields": sorted(list(covered)),
        "rows": rows
    }

# -----------------------------
# Run dataset-level evaluation
# -----------------------------
SUPPORT_THRESHOLD = 0.18  # tune if needed; report in paper
summary_rows = []
sentence_rows = []

for plant_key, info in plant_data.items():
    text = generate_human_like_explanation(plant_key)

    rep = evaluate_one_explanation(
        plant_key=plant_key,
        info=info,
        text=text,
        fields=EVIDENCE_FIELDS,
        support_threshold=SUPPORT_THRESHOLD
    )

    summary_rows.append({
        "plant_key": plant_key,
        "scientific_name": info.get("Scientific Name", ""),
        "faithfulness": rep["faithfulness"],
        "hallucination_rate": rep["hallucination_rate"],
        "coverage": rep["coverage"],
        "n_sentences": rep["sentences"],
        "n_supported_sentences": rep["supported_sentences"]
    })

    sentence_rows.extend(rep["rows"])

summary_df = pd.DataFrame(summary_rows)
sent_df = pd.DataFrame(sentence_rows)

# Save CSVs
summary_path = os.path.join(OUT_DIR, "answer_eval_summary.csv")
sent_path    = os.path.join(OUT_DIR, "answer_eval_sentences.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8")
sent_df.to_csv(sent_path, index=False, encoding="utf-8")

# Dataset stats (Mean ± SD + 95% CI)
f_m, f_sd, f_ci = mean_sd_ci95(summary_df["faithfulness"].dropna())
c_m, c_sd, c_ci = mean_sd_ci95(summary_df["coverage"].dropna())
h_m, h_sd, h_ci = mean_sd_ci95(summary_df["hallucination_rate"].dropna())

stats_df = pd.DataFrame([{
    "Support_threshold": SUPPORT_THRESHOLD,
    "Faithfulness_mean": f_m, "Faithfulness_sd": f_sd, "Faithfulness_CI95_low": f_ci[0], "Faithfulness_CI95_high": f_ci[1],
    "Coverage_mean": c_m, "Coverage_sd": c_sd, "Coverage_CI95_low": c_ci[0], "Coverage_CI95_high": c_ci[1],
    "Hallucination_mean": h_m, "Hallucination_sd": h_sd, "Hallucination_CI95_low": h_ci[0], "Hallucination_CI95_high": h_ci[1],
    "N_plants": int(len(summary_df))
}])

stats_path = os.path.join(OUT_DIR, "answer_eval_stats_ci95.csv")
stats_df.to_csv(stats_path, index=False)

# Alignment matrix (field attribution counts)
# (confusion-matrix-like: which evidence field supports sentences most often)
attrib = sent_df[sent_df["supported"] == True]["best_field"].value_counts()
attrib_path = os.path.join(OUT_DIR, "answer_eval_field_attribution.csv")
attrib.to_csv(attrib_path)

# Plots (high DPI)
def save_hist(col, title, fname):
    plt.figure(figsize=(7, 4), dpi=DPI)
    plt.hist(summary_df[col].dropna().values, bins=10)
    plt.title(title)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=DPI, bbox_inches="tight")
    plt.show()

save_hist("faithfulness", "Faithfulness Distribution (Supported Sentence Ratio)", "faithfulness_hist.png")
save_hist("coverage", "Coverage Distribution (Required Fields Covered)", "coverage_hist.png")
save_hist("hallucination_rate", "Hallucination Rate Distribution", "hallucination_hist.png")

# Save attribution bar plot
plt.figure(figsize=(10, 4), dpi=DPI)
attrib.sort_values(ascending=True).plot(kind="barh")
plt.title("Field Attribution Counts (Supported Sentences)")
plt.xlabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "field_attribution_barh.png"), dpi=DPI, bbox_inches="tight")
plt.show()

print("✅ Saved files:")
for fn in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, fn)
    print(" -", fn, f"({os.path.getsize(fp)} bytes)")

print("\n✅ Dataset-level (journal) stats:")
display(stats_df)
